# Модуль 7. Мониторинг моделей: Data Drift и Concept Drift

**Длительность:** 90 минут
**Формат:** теория (65 мин) + практика (25 мин)
**Цель модуля:** Учащийся должен строго различать Data Drift и Concept Drift на уровне формул вероятности, понимать, почему мониторинг в продакшене вынужден опираться на распределение признаков как на прокси-сигнал (а не на прямую метрику качества), уметь вывести и реализовать PSI с нуля на NumPy, знать его связь с дивергенцией Кульбака-Лейблера, и понимать практические ограничения альтернативы — KS-теста.

## 1. Введение: модель может «тихо» сломаться (5 мин)

### 1.1. Проблема отложенной обратной связи
Все метрики Модулей 2–4 (ROC-AUC, PR-AUC, Precision, Recall, Total Cost) требуют знания истинной метки $y$. В продакшене истинная метка часто приходит с большой задержкой: подтверждение мошенничества может занять недели (спор с банком-эмитентом карты), подтверждение оттока клиента — месяцы (нужно дождаться, чтобы клиент действительно перестал платить), диагноз в медицине — визита к другому специалисту. HTTP-код ответа сервиса (200 OK) ничего не говорит о том, ухудшилось ли качество предсказаний — сервис может исправно отвечать, при этом предсказывая всё хуже и хуже.

### 1.2. Формулировка задачи модуля
Нужен способ обнаружить, что с моделью, вероятно, что-то не так, **используя только то, что доступно немедленно** — входные признаки $X$ и, опционально, предсказания модели $\hat{y}$ — не дожидаясь прихода настоящих меток $y$.

## 2. Data Drift и Concept Drift: формальное разделение (20 мин)

### 2.1. Разложение совместного распределения
Совместное распределение признаков и целевой переменной раскладывается по правилу условной вероятности двумя эквивалентными способами:
$$P(X, Y) = P(X)\cdot P(Y\mid X) = P(Y) \cdot P(X\mid Y)$$
Деградация модели в продакшене — это, по сути, изменение $P(X,Y)$ по сравнению с тем распределением, на котором модель обучалась. Но конкретный **механизм** изменения может затрагивать разные компоненты этого разложения, и от механизма зависит, что вообще можно сделать.

### 2.2. Data Drift
**Определение:** изменилось распределение входных признаков $P(X)$, при этом связь между признаками и целью $P(Y\mid X)$ осталась прежней.

Модель по-прежнему «права» в своей логике — она просто теперь применяется к другой популяции объектов, чем та, на которой обучалась.

**Примеры:**
- Банк запустил маркетинговую кампанию, привлекшую новый сегмент клиентов (например, значительно моложе обычного) — распределение возраста в продакшене сдвинулось относительно обучающей выборки, но закономерность «молодой клиент с такими-то характеристиками риска ведет себя так-то» не изменилась.
- Сломался или был перекалиброван датчик, физически меняющий шкалу одного из признаков (например, единицы измерения суммы транзакции внезапно стали указываться в другой валюте) — распределение признака сдвинулось искусственно, без изменения реальной экономической природы объектов.

### 2.3. Concept Drift
**Определение:** сама зависимость $P(Y\mid X)$ изменилась — при тех же значениях признаков $X$ теперь **иначе** распределена целевая переменная $Y$. Распределение $P(X)$ при этом может как измениться, так и остаться прежним.

**Примеры:**
- Мошенники меняют схему атаки: транзакции с ровно тем же профилем признаков (сумма, время суток, категория товара), которые полгода назад были почти всегда легитимны, теперь всё чаще оказываются мошенническими. Признаки те же — связь с меткой другая.
- Макроэкономический кризис или пандемия: заемщики с одинаковым набором признаков кредитной истории теперь объективно чаще уходят в дефолт, чем раньше, потому что изменились реальные экономические условия, а не характеристики самих заемщиков.

### 2.4. Оба явления могут происходить одновременно и независимо
Data Drift и Concept Drift — ортогональные, не исключающие друг друга явления. Возможна ситуация, когда распределение признаков не изменилось вовсе ($P(X)$ то же самое), но связь с меткой изменилась радикально (чистый Concept Drift) — и наоборот, признаки могут заметно «поплыть», а истинная зависимость остаться прежней (чистый Data Drift). На практике чаще всего наблюдается смесь обоих эффектов, и различить их количественно, не имея размеченных данных из продакшена, в общем случае невозможно.

## 3. Почему мониторинг $P(X)$ — единственный доступный прокси без меток (10 мин)

### 3.1. Ограничение доступных сигналов
Без истинных меток $Y$ у нас нет прямого способа измерить $P(Y\mid X)$ на продакшен-данных — соответственно, Concept Drift **напрямую** не наблюдаем в реальном времени. Единственное, что физически доступно сразу — это сами признаки $X$ каждого нового объекта (и, опционально, выход модели $\hat{Y}=f(X)$, вычисляемый в момент инференса).

### 3.2. Почему мониторинг $P(X)$ — это компромисс, а не полное решение
Отслеживание сдвига $P(X)$ напрямую обнаруживает Data Drift. Но оно лишь **косвенно** сигнализирует о возможном Concept Drift, и делает это неполно: если Concept Drift произошел без сопутствующего сдвига распределения признаков (мошенники начали использовать ровно те же паттерны признаков, но с другим реальным исходом), мониторинг $P(X)$ **ничего не заметит** — с точки зрения входных данных всё выглядит стабильно, хотя модель уже дает систематически неверные предсказания.

### 3.3. Дополнительный сигнал: распределение выходов модели $P(\hat{Y})$
Частичное смягчение этой слепой зоны — параллельный мониторинг распределения **предсказаний** модели ($\hat{p} = f(x)$ для потока продакшен-объектов), а не только входных признаков. Если решающая граница модели фактически стала неточной из-за Concept Drift, это иногда (хотя не всегда и не гарантированно) проявляется как сдвиг в распределении выдаваемых моделью вероятностей — например, доля объектов, получающих скор выше определенного порога, начинает заметно отклоняться от исторической нормы. Это не заменяет отслеживание $P(X)$, а дополняет его — оба сигнала используются одновременно в зрелых системах мониторинга.

### 3.4. Итоговая картина ограничений
Мониторинг без меток — это система раннего оповещения с заведомо неполным покрытием: она надежно ловит Data Drift, частично и не гарантированно намекает на некоторые формы Concept Drift через $P(\hat{Y})$, но принципиально не может заменить периодическую (пусть и отложенную) проверку модели на реальных, дождавшихся разметки данных. Осознание этого ограничения — важная часть зрелого инженерного понимания темы, а не только умение посчитать формулу PSI.

## 4. PSI: конструкция и формула (20 мин)

### 4.1. Общая идея
**Population Stability Index (PSI)** — числовая мера того, насколько распределение одного признака в текущих (продакшен, Actual) данных отличается от распределения этого же признака в эталонных (обучающих, Expected) данных.

### 4.2. Построение бинов
1. Признак разбивается на $K$ интервалов (бинов), обычно $K=10$.
2. **Критически важная деталь:** границы бинов строятся **по квантилям эталонного (Expected/обучающего) распределения**, а не по объединенным данным и не по продакшен-данным. Это гарантирует, что в самой эталонной выборке каждый бин по построению содержит ровно $1/K$ (например, 10 %) наблюдений — равномерную «линейку», по которой затем измеряется искажение.
3. Те же самые границы применяются к продакшен-данным (Actual), и для них подсчитывается, какая доля объектов попала в каждый бин — в общем случае **не** равная $1/K$, если распределение сдвинулось.

### 4.3. Формула
Для каждого бина $i$:
$$PSI = \sum_{i=1}^{K}\left(\text{Actual}\%_i - \text{Expected}\%_i\right)\cdot\ln\!\left(\frac{\text{Actual}\%_i}{\text{Expected}\%_i}\right)$$
где $\text{Expected}\%_i$ — доля обучающих объектов в бине $i$ (по построению $\approx 1/K$ для каждого бина), $\text{Actual}\%_i$ — доля продакшен-объектов в том же бине.

### 4.4. Почему именно такая формула: связь с дивергенцией Кульбака-Лейблера
Раскроем сумму алгебраически:
$$PSI = \sum_i \text{Actual}\%_i\ln\frac{\text{Actual}\%_i}{\text{Expected}\%_i} - \sum_i \text{Expected}\%_i\ln\frac{\text{Actual}\%_i}{\text{Expected}\%_i}$$
Первое слагаемое — это по определению дивергенция Кульбака-Лейблера от Expected к Actual:
$$\sum_i \text{Actual}\%_i\ln\frac{\text{Actual}\%_i}{\text{Expected}\%_i} = D_{KL}(\text{Actual}\,\|\,\text{Expected})$$
Второе слагаемое, если переставить знак под логарифмом ($\ln(a/e) = -\ln(e/a)$):
$$-\sum_i \text{Expected}\%_i\ln\frac{\text{Actual}\%_i}{\text{Expected}\%_i} = \sum_i \text{Expected}\%_i\ln\frac{\text{Expected}\%_i}{\text{Actual}\%_i} = D_{KL}(\text{Expected}\,\|\,\text{Actual})$$
Значит:
$$PSI = D_{KL}(\text{Actual}\,\|\,\text{Expected}) + D_{KL}(\text{Expected}\,\|\,\text{Actual})$$

**PSI — это в точности сумма KL-дивергенции в обе стороны, известная в теории информации как дивергенция Джеффриса (Jeffreys divergence), или симметризованная KL-дивергенция.** Обычная KL-дивергенция несимметрична ($D_{KL}(A\|E) \ne D_{KL}(E\|A)$ в общем случае), что неудобно для показателя «насколько два распределения различаются» — не очевидно, какое из двух распределений ставить первым. Симметризация через сложение обеих версий устраняет эту произвольность. Это не просто удачно придуманная эвристическая формула — она имеет точный смысл в теории информации, что полезно знать для технического собеседования, даже если сама библиотечная реализация PSI никогда явно не апеллирует к KL-дивергенции.

### 4.5. Почему бины строятся именно по квантилям Expected
Если бы границы бинов строились, например, равномерно по диапазону значений (равные по ширине интервалы, а не по населенности), небольшой сдвиг в «плотной» части распределения (где сосредоточено большинство наблюдений) и такой же по величине сдвиг в «разреженной» части (хвост распределения) вносили бы несопоставимый вклад в PSI, искажая интерпретацию. Квантильные бины эталонного распределения гарантируют, что каждый бин изначально содержит одинаковую «статистическую массу», что делает вклад каждого бина в PSI сопоставимым по порядку величины.

## 5. Проблема нулевых бинов и сглаживание (10 мин)

### 5.1. Формулировка проблемы

Если в каком-то бине $\text{Actual}\%_i = 0$ (в продакшен-потоке за анализируемый период вообще не оказалось объектов в этом диапазоне значений признака) или $\text{Expected}\%_i = 0$ (что реже, но возможно при неаккуратном построении границ), формула ломается:
- $\ln(0) = -\infty$;
- деление на ноль внутри логарифма, если $\text{Expected}\%_i=0$, дает неопределенность.

На практике это означает: без защиты функция `calculate_psi` может либо упасть с ошибкой, либо тихо вернуть `NaN` или `inf`, что в системе автоматического мониторинга — гораздо хуже, чем просто неточное число: пайплайн мониторинга может незаметно перестать работать.

### 5.2. Решение: сглаживание
Стандартная защита — заменить точный ноль на маленькое положительное число $\varepsilon$ (например, $10^{-5}$ или $10^{-6}$) **перед** вычислением логарифма и отношения:
$$\text{Actual}\%_i \leftarrow \max(\text{Actual}\%_i,\ \varepsilon), \qquad \text{Expected}\%_i \leftarrow \max(\text{Expected}\%_i,\ \varepsilon)$$
Это разновидность сглаживания Лапласа — та же идея, что используется, например, при вычислении условных вероятностей в наивном байесовском классификаторе на разреженных данных.

### 5.3. Почему это приемлемое, а не «грязное» решение
Замена нуля на $\varepsilon = 10^{-5}$ (на несколько порядков меньше типичной доли бина $1/K \approx 0.1$) практически не искажает вклад ненулевых бинов, но превращает потенциально катастрофический (бесконечный или неопределенный) вклад пустого бина в очень большое, но конечное число — что и требуется: пустой в продакшене бин **должен** сильно повышать PSI (это действительно тревожный сигнал — целый диапазон значений признака, который раньше встречался, исчез), но не должен ломать вычисление целиком.

## 6. Интерпретация PSI и её ограничения (10 мин)

### 6.1. Пороговые значения (индустриальная конвенция)

| Значение PSI | Интерпретация |
|---|---|
| $< 0.1$ | стабильно, значимого сдвига нет |
| $0.1 \le PSI \le 0.25$ | умеренный сдвиг — требует наблюдения, повод присмотреться |
| $> 0.25$ | критический дрейф — веский повод для пересмотра/переобучения модели |

### 6.2. Важная оговорка
Эти пороги — не результат математического вывода, а **эмпирическая индустриальная конвенция**, сложившаяся в первую очередь в практике кредитного скоринга (откуда PSI и происходит исторически). Ничто не мешает конкретной компании откалибровать собственные пороги под свою специфику, сопоставив исторические значения PSI с реально наблюдавшейся деградацией бизнес-метрик модели за прошлые периоды. На собеседовании стоит явно проговорить: «пороги 0.1/0.25 — общепринятые ориентиры, а не физическая константа».

### 6.3. Ограничения метода
- **Чувствительность к выбору $K$.** Разное число бинов может давать заметно разное значение PSI для одних и тех же данных, особенно при малых выборках, где на бин попадает мало наблюдений — это делает PSI менее объективным сравнительным инструментом, чем хотелось бы, и требует фиксировать $K$ заранее, а не подбирать постфактум.
- **PSI видит только одномерные (маргинальные) сдвиги по каждому признаку отдельно.** Он не обнаруживает изменение **совместного** распределения нескольких признаков (например, если раньше высокая сумма транзакции сочеталась только с определенным типом товара, а теперь эта связь нарушилась, при том что маргинальные распределения суммы и типа товара по отдельности не изменились) — для этого нужны отдельные многомерные методы обнаружения дрейфа, выходящие за рамки этого модуля.
- **PSI — мера величины сдвига, а не статистической значимости.** Он не говорит, насколько мы уверены, что сдвиг реален, а не случайная флуктуация выборки — для строгой статистической проверки существует другой инструмент, разбираемый в следующем разделе.

## 7. KS-тест (Колмогорова-Смирнова) (15 мин)

### 7.1. Определение
Двухвыборочный критерий Колмогорова-Смирнова проверяет нулевую гипотезу $H_0$: «две выборки взяты из одного и того же распределения». Статистика критерия:
$$D = \sup_x \left|F_{\text{expected}}(x) - F_{\text{actual}}(x)\right|$$
— максимальное по модулю расхождение между эмпирическими функциями распределения (ECDF) двух выборок в любой точке $x$. Чем больше $D$, тем сильнее выборки расходятся хоть в какой-то точке своего распределения.

### 7.2. Принципиальные отличия от PSI

| | PSI | KS-тест |
|---|---|---|
| Результат | Число (величина сдвига), интерпретируемое по эмпирическим порогам | Статистика $D$ и p-value (формальная статистическая значимость) |
| Требует ли бинов | Да, явно | Нет — работает с полной эмпирической функцией распределения, не дискретизируя данные |
| Что измеряет | Насколько сильно распределение сдвинулось (по величине) | Достаточно ли доказательств, чтобы отвергнуть гипотезу «распределения одинаковы» |
| Индустриальная традиция интерпретации порогов | Да (0.1 / 0.25) | Нет привычной «бизнес-шкалы» — только стандартный порог значимости p-value (обычно 0.05) |

### 7.3. Критическая практическая ловушка: мощность теста растет с размером выборки
Статистическая мощность любого гипотезного теста, в том числе KS, растет с объемом выборки: чем больше наблюдений, тем меньшее по величине реальное различие между распределениями тест способен «уверенно» обнаружить. Следствие для продакшена: если ежедневно через мониторинг проходят миллионы транзакций, KS-тест почти неизбежно будет отвергать нулевую гипотезу (p-value $\to 0$) даже при исчезающе малом, практически незначимом сдвиге распределения — просто потому, что при таком объеме данных статистика способна «уловить» даже шум измерения или естественные сезонные колебания как «статистически значимое» отличие.

**Практическое следствие:** на больших продакшен-потоках p-value KS-теста быстро перестает быть содержательным индикатором (он почти всегда близок к нулю), в то время как PSI, будучи нормированным по долям, а не по абсолютным счетчикам, сохраняет содержательную интерпретацию независимо от объема выборки. Это ключевая практическая причина, по которой индустрия для мониторинга по расписанию чаще опирается именно на PSI, а KS-тест используется как дополнительный диагностический инструмент, а не как основной триггер алертинга.

## 8. Практика: `calculate_psi` на чистом NumPy и сравнение с `ks_2samp` (25 мин)

### 8.1. Постановка задачи
Реализовать `calculate_psi(expected, actual, num_bins=10)` с защитой от нулевых бинов. Сгенерировать `expected ~ N(0, 1)` и `actual ~ N(0.2, 1.1)` (сдвиг среднего и рост дисперсии — правдоподобная модель умеренного дрейфа), сравнить PSI и `ks_2samp`. Дополнительно — продемонстрировать зависимость чувствительности KS-теста от размера выборки (раздел 7.3) на числах.

### 8.2. Код: `calculate_psi`

In [ ]:
import numpy as np
from scipy.stats import ks_2samp

def calculate_psi(expected, actual, num_bins=10, epsilon=1e-5):
    """
    Population Stability Index между двумя одномерными выборками.

    Параметры
    ---------
    expected : array-like — эталонное (обучающее) распределение признака
    actual   : array-like — текущее (продакшен) распределение признака
    num_bins : int — число квантильных бинов, построенных по expected
    epsilon  : float — сглаживание против log(0) / деления на ноль

    Возвращает
    ----------
    psi          : float — итоговое значение PSI
    psi_per_bin  : np.ndarray — вклад каждого бина (для диагностики)
    bin_edges    : np.ndarray — границы бинов
    """
    expected = np.asarray(expected)
    actual = np.asarray(actual)

    # Границы бинов строятся строго по expected-распределению (раздел 4.2)
    quantiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(expected, quantiles)
    bin_edges[0] = -np.inf   # защита от значений actual за пределами диапазона expected
    bin_edges[-1] = np.inf

    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)

    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    # Сглаживание нулевых бинов (раздел 5.2)
    expected_pct = np.where(expected_pct == 0, epsilon, expected_pct)
    actual_pct = np.where(actual_pct == 0, epsilon, actual_pct)

    psi_per_bin = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    psi = np.sum(psi_per_bin)

    return psi, psi_per_bin, bin_edges


# 1. Симуляция умеренного дрейфа
np.random.seed(42)
expected = np.random.normal(loc=0.0, scale=1.0, size=10_000)
actual = np.random.normal(loc=0.2, scale=1.1, size=10_000)

psi_value, psi_per_bin, bin_edges = calculate_psi(expected, actual, num_bins=10)

print(f"PSI = {psi_value:.4f}")
if psi_value < 0.1:
    verdict = "стабильно"
elif psi_value < 0.25:
    verdict = "умеренный сдвиг — наблюдать"
else:
    verdict = "критический дрейф — пересматривать модель"
print(f"Интерпретация: {verdict}")

print("\nВклад каждого бина в PSI:")
for i, contrib in enumerate(psi_per_bin):
    print(f"  Бин {i+1}: {contrib:+.4f}")

# 2. KS-тест для того же сравнения
ks_stat, ks_pvalue = ks_2samp(expected, actual)
print(f"\nKS-статистика: {ks_stat:.4f}")
print(f"p-value:       {ks_pvalue:.2e}")

### 8.3. Код: демонстрация зависимости KS-теста от размера выборки

In [ ]:
print("\n--- Чувствительность KS-теста к объему выборки (раздел 7.3) ---")
print(f"{'n':>8} | {'PSI':>8} | {'KS statistic':>12} | {'KS p-value':>12}")

for n in [50, 500, 5_000, 50_000, 500_000]:
    exp_n = np.random.normal(0.0, 1.0, n)
    act_n = np.random.normal(0.05, 1.02, n)   # очень небольшой, практически малозначимый сдвиг
    psi_n, _, _ = calculate_psi(exp_n, act_n, num_bins=10)
    ks_stat_n, ks_p_n = ks_2samp(exp_n, act_n)
    print(f"{n:>8} | {psi_n:>8.4f} | {ks_stat_n:>12.4f} | {ks_p_n:>12.4g}")

### 8.4. Что должно получиться и как интерпретировать
- Для основного сценария (сдвиг среднего на 0.2, роста std до 1.1, $n=10\,000$) ожидается PSI в диапазоне примерно 0.1–0.3 (умеренный или близкий к критическому дрейф — правдоподобно для заданного по условию сдвига) и KS-тест с исчезающе малым p-value (сдвиг легко статистически значим при таком объеме выборки).
- В таблице зависимости от $n$ (раздел 8.3) ожидается: **PSI остается в целом стабильным порядка величины** при росте $n$ (небольшие колебания за счет случайности выборки и квантования по бинам), в то время как **p-value KS-теста стремительно падает** с ростом $n$, становясь исчезающе малым уже при $n$ порядка десятков тысяч — несмотря на то что реальная величина сдвига (0.05 по среднему, 0.02 по std) остается одной и той же на всех строчках таблицы. Это прямая, наглядная числовая иллюстрация практической ловушки из раздела 7.3.
- Обсуждение вслух: если бы решение «перезапускать переобучение модели» принималось автоматически по правилу «p-value KS-теста < 0.05», при большом продакшен-трафике модель переобучалась бы практически ежедневно из-за статистически значимых, но практически несущественных колебаний — отсюда практический вывод в пользу PSI как основного триггера мониторинга.

## 9. Итоги модуля (5 мин)

### Ключевые тезисы
1. Data Drift — сдвиг $P(X)$ при неизменной $P(Y\mid X)$; Concept Drift — изменение самой зависимости $P(Y\mid X)$, независимо от того, сдвинулось ли $P(X)$.
2. Без истинных меток в реальном времени мониторинг вынужденно опирается на $P(X)$ (и, дополнительно, на $P(\hat{Y})$) как на неполный прокси-сигнал — чистый Concept Drift без сопутствующего сдвига признаков этим способом не обнаруживается.
3. PSI строится через квантильные бины эталонного (обучающего) распределения и по формуле $\sum(\text{Actual}\%-\text{Expected}\%)\ln(\text{Actual}\%/\text{Expected}\%)$ является в точности суммой KL-дивергенции в обе стороны — симметризованной дивергенцией Джеффриса.
4. Нулевые бины требуют сглаживания малым $\varepsilon$, иначе формула дает неопределенность или бесконечность.
5. Пороги интерпретации PSI (0.1 / 0.25) — эмпирическая индустриальная конвенция, не математически выведенная константа.
6. KS-тест дает формальную статистическую значимость различия распределений, но его мощность растет с объемом выборки: на больших продакшен-потоках p-value почти всегда исчезающе мал, что делает его ненадежным единственным триггером алертинга по сравнению с PSI.

### Контрольные вопросы
- Приведите пример ситуации, в которой происходит чистый Concept Drift без какого-либо Data Drift, и объясните, почему мониторинг $P(X)$ его не заметит.
- Выведите тождество $PSI = D_{KL}(\text{Actual}\|\text{Expected}) + D_{KL}(\text{Expected}\|\text{Actual})$ из исходной формулы PSI.
- Почему границы бинов для расчета PSI строятся по квантилям именно эталонного (Expected), а не продакшен (Actual) распределения?
- Почему для очень большого продакшен-потока p-value KS-теста перестает быть надежным сигналом для принятия решения о переобучении модели?
- Какие два ограничения PSI важно проговорить на собеседовании, чтобы не создать впечатление, что PSI — универсальный и безупречный инструмент мониторинга?

### Что дальше
В следующем модуле мы вернемся к вопросу, поднятому еще в Модуле 5 (Probability Shifting) и частично затронутому в разделе 3.3 этого модуля: как превратить сырые, потенциально искаженные скоры модели в честные физические вероятности — через Reliability Diagram, Platt Scaling и Isotonic Regression.